Identifying **customer sentiment and intent** - to showcase how the technical approach evolved from manual rules to statistical machine learning, and finally to semantic LLM understanding.

### Prerequisites

Before running the script, install the required libraries:

*(To run the GenAI part locally, ensure you have [Ollama](https://ollama.com/) running with a model like `ollama run llama3`, or replace the placeholder function with your OpenAI/Anthropic API key).*

In [ ]:
!pip install scikit-learn

In [ ]:
import re
import time
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.naive_bayes import MultinomialNB

In [ ]:
test_cases = [
    "This product is absolute trash, it broke on day one.",                     # Simple Negative
    "There is no way this product is bad, it's absolutely amazing!",           # Complex/Sarcastic (Fails Rule-Based)
    "Can someone tell me how to reset the password?",                          # Out of Domain / Query (Fails ML)
]

### APPROACH 1: RULE-BASED SYSTEM (1970s - 1990s)


In [ ]:
def rule_based_sentiment(text):
    # Hardcoded keyword matching rules
    negative_words = r'\b(bad|trash|broken|terrible|worst|fail)\b'
    positive_words = r'\b(good|amazing|great|awesome|love|best)\b'

    text_lower = text.lower()

    if re.search(negative_words, text_lower):
        return "Negative"
    elif re.search(positive_words, text_lower):
        return "Positive"
    else:
        return "Neutral / Unknown"

In [ ]:
print("\n[APPROACH 1] Running Rule-Based System...")
time.sleep(1)
for i, text in enumerate(test_cases, 1):
    result = rule_based_sentiment(text)
    print(f"  Test {i}: '{text}'")
    print(f"  Predicted Sentiment: {result}\n")


[APPROACH 1] Running Rule-Based System...
  Test 1: 'This product is absolute trash, it broke on day one.'
  Predicted Sentiment: Negative

  Test 2: 'There is no way this product is bad, it's absolutely amazing!'
  Predicted Sentiment: Negative

  Test 3: 'Can someone tell me how to reset the password?'
  Predicted Sentiment: Neutral / Unknown



### APPROACH 2: MACHINE LEARNING (2000s - 2010s)

In [ ]:
# 1. Training Data (ML needs data history to learn patterns)
training_corpus = [
    "I love this product, it is amazing", "Positive",
    "Great customer service and good quality", "Positive",
    "This is the best purchase I have ever made", "Positive",
    "Horrible experience, completely broken", "Negative",
    "The item is trash and a waste of money", "Negative",
    "Terrible quality, it failed immediately", "Negative"
]
X_train = training_corpus[0::2]
y_train = training_corpus[1::2]

In [ ]:
# 2. Vectorization (Turning text into mathematical frequency matrices)
vectorizer = CountVectorizer()
X_train_vectorized = vectorizer.fit_transform(X_train)

In [ ]:
# 3. Train a Naive Bayes Classifier
ml_model = MultinomialNB()
ml_model.fit(X_train_vectorized, y_train)

MultinomialNB()

In [ ]:
def machine_learning_sentiment(text):
    # Convert input text into the same math matrix format
    text_vectorized = vectorizer.transform([text])
    prediction = ml_model.predict(text_vectorized)
    return prediction[0]

In [ ]:
print("-" * 70)
print("[APPROACH 2] Running Machine Learning (Naive Bayes)...")
time.sleep(1)
for i, text in enumerate(test_cases, 1):
    result = machine_learning_sentiment(text)
    print(f"  Test {i}: '{text}'")
    print(f" Predicted Sentiment: {result}\n")

----------------------------------------------------------------------
[APPROACH 2] Running Machine Learning (Naive Bayes)...
  Test 1: 'This product is absolute trash, it broke on day one.'
 Predicted Sentiment: Positive

  Test 2: 'There is no way this product is bad, it's absolutely amazing!'
 Predicted Sentiment: Positive

  Test 3: 'Can someone tell me how to reset the password?'
 Predicted Sentiment: Negative



# APPROACH 3: GENERATIVE AI / LLM (2020s+)

### Step 1: Install Dependencies
Install the required packages for LangGraph and LangChain Ollama integration.

In [ ]:
!pip install -q langgraph langchain-core langchain-ollama

### Step 2: Configure Ollama Background Process
Launch the Ollama server service in the background and download the open-weight `gpt-oss` model.

In [ ]:
!sudo apt-get update && sudo apt-get install -y zstd

Get:1 https://cloud.r-project.org/bin/linux/ubuntu jammy-cran40/ InRelease [3,632 B]
Get:2 https://cli.github.com/packages stable InRelease [3,917 B]
Get:3 http://security.ubuntu.com/ubuntu jammy-security InRelease [129 kB]
Get:4 https://cli.github.com/packages stable/main amd64 Packages [354 B]
Hit:5 http://archive.ubuntu.com/ubuntu jammy InRelease
Get:6 http://security.ubuntu.com/ubuntu jammy-security/restricted amd64 Packages [7,090 kB]
Get:7 http://security.ubuntu.com/ubuntu jammy-security/universe amd64 Packages [1,297 kB]
Get:8 http://archive.ubuntu.com/ubuntu jammy-updates InRelease [128 kB]
Get:9 http://security.ubuntu.com/ubuntu jammy-security/main amd64 Packages [3,959 kB]
Get:10 https://r2u.stat.illinois.edu/ubuntu jammy InRelease [6,555 B]
Hit:11 https://ppa.launchpadcontent.net/deadsnakes/ppa/ubuntu jammy InRelease
Hit:12 https://ppa.launchpadcontent.net/ubuntugis/ppa/ubuntu jammy InRelease
Get:13 https://r2u.stat.illinois.edu/ubuntu jammy/main amd64 Packages [3,032 kB]
Ge

In [ ]:
!curl -fsSL https://ollama.com/install.sh | sh

>>> Installing ollama to /usr/local
>>> Downloading ollama-linux-amd64.tar.zst
######################################################################## 100.0%
>>> Creating ollama user...
>>> Adding ollama user to video group...
>>> Adding current user to ollama group...
>>> Creating ollama systemd service...
>>> The Ollama API is now available at 127.0.0.1:11434.
>>> Install complete. Run "ollama" from the command line.


In [ ]:
!nohup ollama serve > ollama.log &

nohup: redirecting stderr to stdout


In [ ]:
import os, subprocess, time, requests, json, re
os.environ['OLLAMA_HOST'] = '0.0.0.0:11434'
subprocess.Popen(["ollama", "serve"], stdout=subprocess.PIPE, stderr=subprocess.PIPE)
time.sleep(10)
print("Ollama server should be ready on http://localhost:11434")

Ollama server should be ready on http://localhost:11434


In [ ]:
!ollama pull gpt-oss

In [ ]:
!ollama pull gemma4:e2b

In [ ]:
OLLAMA_API_URL = "http://localhost:11434/api/generate"

def call_ollama(prompt, model="gemma4:e2b", stream=False):
    payload = {"model": model, "prompt": prompt, "stream": stream}
    resp = requests.post(OLLAMA_API_URL, json=payload)
    if resp.status_code != 200:
        raise RuntimeError(f"Request failed: {resp.status_code}, {resp.text}")
    if stream:
        for line in resp.text.splitlines():
            if not line.strip():
                continue
            obj = json.loads(line)
            print(obj.get("response", ""), end="")
    else:
        data = resp.json()
        return data.get("response", "")

In [ ]:
import json
import requests

def genai_sentiment_and_intent(text):
    """
    Leverages a local Ollama instance running gemma4:e2b to perform
    deep semantic understanding, handling nuance and multiple reasoning tasks.
    """
    OLLAMA_API_URL = "http://localhost:11434/api/generate"

    # Craft a structured prompt to ensure the model outputs clean JSON
    prompt = f"""
    You are an advanced customer support AI. Analyze the following customer text for:
    1. Sentiment (Positive, Negative, or Neutral)
    2. Intent or Nuance (Brief description of what the user wants or the underlying tone)
    3. Action (Recommended next step, e.g., 'Tag for testimonial review', 'Route to IT Helpdesk', 'Generate apology draft & trigger refund')

    Respond ONLY with a valid JSON object matching this structure:
    {{
        "Sentiment": "...",
        "Intent": "...",
        "Action": "..."
    }}

    Customer text: "{text}"
    """

    payload = {
        "model": "gemma4:e2b",
        "prompt": prompt,
        "stream": False,
        "format": "json" # Forces the model to return valid JSON
    }

    try:
        resp = requests.post(OLLAMA_API_URL, json=payload)
        if resp.status_code == 200:
            # Extract the text response from Ollama's JSON wrapper
            data = resp.json()
            return data.get("response", "").strip()
        else:
            return f"{{\n    'Error': 'Ollama request failed with status {resp.status_code}'\n}}"
    except Exception as e:
        return f"{{\n    'Error': 'Could not connect to Ollama: {str(e)}'\n}}"

In [ ]:
print("[APPROACH 3] Running Generative AI (Semantic Context)...")
time.sleep(1)
for i, text in enumerate(test_cases, 1):
    result = genai_sentiment_and_intent(text)
    print(f"  Test {i}: '{text}'")
    print(f"  LLM Contextual JSON Output:\n{result}\n")

[APPROACH 3] Running Generative AI (Semantic Context)...
  Test 1: 'This product is absolute trash, it broke on day one.'
  LLM Contextual JSON Output:
{
    "Sentiment": "Negative",
    "Intent": "Expressing extreme dissatisfaction and frustration with a product that failed immediately.",
    "Action": "Route to Customer Service for immediate issue resolution (replacement/refund) and internal quality assurance investigation."
}

  Test 2: 'There is no way this product is bad, it's absolutely amazing!'
  LLM Contextual JSON Output:
{
    "Sentiment": "Positive",
    "Intent": "Expressing strong satisfaction and enthusiasm regarding the product.",
    "Action": "Log feedback as positive review/compliment."
}

  Test 3: 'Can someone tell me how to reset the password?'
  LLM Contextual JSON Output:
{
    "Sentiment": "Neutral",
    "Intent": "Seeking instructions for a specific task (password reset)",
    "Action": "Provide step-by-step instructions for password reset"
}

